In [1]:
import os
import torch
from samples_setup import * 
from torchvision import transforms


### Environment set up


In [2]:
# specify GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

print(torch.cuda.get_device_name(0))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Specify paths
data_directory = '/data/WHOI-Plankton'
data_subdirectories = ['2010','2011', '2012','2013','2014']

# Specify other environment variables
SEED = 666
set_seed(SEED)

NVIDIA GeForce RTX 4090
Using device: cuda


### Data preparation

In [3]:

PLANKTON_CLASSES = [
    'Cylindrotheca',
    'Cerataulina',
    'Chaetoceros',
    'Chaetoceros_didymus_flagellate',
    'Corethron',
    'Dactyliosolen',
    'Ditylum',
    'Eucampia',
    'Guinardia_delicatula',
    'Guinardia_striata',
    'G_delicatula_external_parasite',
    'Leptocylindrus',
    'Pseudonitzschia',
    'Rhizosolenia',
    'Skeletonema',
    'Thalassiosira',
    'Thalassionema'         
    ]

dataset_selected = ImageDataset(
    data_directory = data_directory,
    data_subdirectories = data_subdirectories,
    class_names = PLANKTON_CLASSES,
    #max_class_size = 5000,
    image_resolution = 64,
    image_transforms = None,
    format_file = '.png',
    seed = SEED
    )

# Merge categories
classes_to_merge_list = [
    [
    'Guinardia_delicatula',
    'Guinardia_striata',
    'G_delicatula_external_parasite',
   ],
   [
    'Chaetoceros',
    'Chaetoceros_didymus_flagellate',   
   ]
]
new_names_list = [
    'Guinardia',
    'Chaetoceros'
]

# Define final dataset
dataset =  merge_classes(
    dataset = dataset_selected,
    classes_to_merge_list=classes_to_merge_list,
    new_names_list=new_names_list
    )

NUM_CLASSES = len(dataset.class_ids)

In [4]:
# Add Image Transformations to Pipeline
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.Pad(padding = 5, fill = 0),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])


dataset.append_image_transforms(
    image_transforms = train_transforms, verbose = False, replace = True
)

dataset.print_image_transforms()


Current Image Transform Pipeline:
  RandomHorizontalFlip(p=0.5)
  RandomVerticalFlip(p=0.5)
  RandomRotation(degrees=[-180.0, 180.0], interpolation=nearest, expand=False, fill=0)
  Pad(padding=5, fill=0, padding_mode=constant)
  Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
  ToTensor()


In [5]:
# Split data into train, test and validation

TRAIN_PROP = 0.7
VAL_PROP = 0.1
TEST_PROP = 0.2

BATCH_SIZE = 64


train_split, val_split, test_split = dataset.split_train_test_val(
    train_prop = TRAIN_PROP, val_prop = VAL_PROP, test_prop = TEST_PROP, verbose = False
)